# UFC Unsupervised ML — Fighter Archetypes & Stat Relationships

This notebook walks the same flow as the reusable scripts (`db.py`, `archetypes.py`, `relationships.py`) but renders charts inline.

**Data source:** `data/ufc.db`, a local SQLite database produced by the Go scraper (`scraper-go/`). This notebook is **read-only** on that database.

Flow: load tables -> engineer per-fighter features -> PCA + clustering (archetypes) -> archetype table -> correlation heatmap -> association rules.

If `data/ufc.db` does not exist yet, run the scraper first.

## Setup - install dependencies

**Recommended (isolated venv):** from the `ml/` directory run `./setup_env.ps1` (Windows) or
`bash setup_env.sh` (macOS/Linux) **once**. It creates `.venv`, installs the requirements, and
registers a Jupyter kernel named **"UFC ML (.venv)"**. Then pick that kernel (top-right) and
skip the next cell.

**Quick (current kernel):** just run the next cell - `%pip` installs into whichever kernel is
running. If imports still fail immediately after, restart the kernel and run it again.

In [ ]:
# Installs this notebook's dependencies into the CURRENTLY RUNNING kernel.
# (Skip this if you selected the .venv kernel created by setup_env.ps1 / setup_env.sh.)
%pip install -q -r requirements.txt

## 0. Setup & imports

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Inline charts in the notebook.
%matplotlib inline
sns.set_theme(style='whitegrid')

# Import our reusable modules (run this notebook from the ml/ dir).
import db
import archetypes
import relationships

DB_PATH = db.DEFAULT_DB_PATH  # ../data/ufc.db relative to ml/
print('DB path:', DB_PATH, '| exists:', os.path.exists(DB_PATH))

ModuleNotFoundError: No module named 'seaborn'

## 1. Load the raw tables

`fighters` (one row per fighter), `fights` (one row per bout), and `round_stats` (one wide row per fight x fighter x round).

In [ ]:
fighters = db.load_fighters(DB_PATH)
fights = db.load_fights(DB_PATH)
round_stats = db.load_round_stats(DB_PATH)

print('fighters   :', fighters.shape)
print('fights     :', fights.shape)
print('round_stats:', round_stats.shape)
fighters.head()

## 2. Engineer the per-fighter feature matrix

`build_fighter_features()` joins career averages, physical attributes, derived features (`win_rate`, `finish_rate`) and per-round aggregates from `round_stats`. NULLs are median-imputed; fighters with too few bouts or too many missing career stats are dropped.

In [ ]:
features = db.build_fighter_features(DB_PATH)
print('feature matrix:', features.shape)
print('features:', list(features.columns))
features.describe().T

## 3. Scale + PCA + clustering (archetypes)

Standardize, reduce with PCA (~95% variance), choose `k` for KMeans by silhouette, and also fit AgglomerativeClustering for the dendrogram.

In [ ]:
X_scaled, X_pca, X_pca2, scaler, pca_full, pca2 = archetypes.scale_and_reduce(features)
print(f'PCA components for ~95% variance: {X_pca.shape[1]} '
      f'({pca_full.explained_variance_ratio_.sum()*100:.1f}% retained)')

best_k, ks, sils, inertias = archetypes.choose_k(X_pca)
print('silhouette-suggested k =', best_k, '(curve is flat -> styles are a continuum)')

# Fighter styles form a continuum, so the silhouette curve is near-flat and
# auto-picks a trivial k=2. Choose k for interpretable archetypes instead.
K = 6
print('using k =', K, 'archetypes')

In [ ]:
# Silhouette + elbow (inertia) vs k, inline.
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(ks, sils, 'o-', color='tab:blue', label='silhouette')
ax1.set_xlabel('k'); ax1.set_ylabel('silhouette', color='tab:blue')
ax1.axvline(best_k, color='green', ls='--', alpha=0.7)
ax2 = ax1.twinx()
ax2.plot(ks, inertias, 's--', color='tab:red', alpha=0.7, label='inertia')
ax2.set_ylabel('inertia (elbow)', color='tab:red')
ax1.set_title('Choosing k: silhouette and inertia')
plt.show()

In [ ]:
from sklearn.cluster import KMeans
km = KMeans(n_clusters=K, n_init=10, random_state=42)
labels = km.fit_predict(X_pca)

# 2-D PCA scatter coloured by cluster.
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_pca2[:, 0], X_pca2[:, 1], c=labels, cmap='tab10', s=28, alpha=0.85)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Fighter archetypes (PCA projection)')
ax.add_artist(ax.legend(*sc.legend_elements(), title='cluster'))
plt.show()

In [ ]:
# Dendrogram (hierarchical view).
from scipy.cluster.hierarchy import dendrogram, linkage
Z = linkage(X_scaled, method='ward')
fig, ax = plt.subplots(figsize=(11, 5))
_ = dendrogram(Z, ax=ax, truncate_mode='lastp', p=30, show_leaf_counts=True)
ax.set_title('Hierarchical clustering dendrogram (Ward)')
ax.set_ylabel('distance')
plt.show()

## 4. Archetype table

Per-cluster feature means plus a suggested human-readable archetype label derived from each cluster's highest-deviation stats.

In [ ]:
profiles = archetypes.build_cluster_profiles(features, labels)
profiles[['size', 'archetype_label']]

In [ ]:
# Full per-cluster feature means.
profiles.drop(columns=['archetype_label'])

## 5. Correlation heatmap

Pearson (linear) and Spearman (monotonic) correlations over the engineered features including `win_rate`.

In [ ]:
pearson, spearman = relationships.correlation_matrices(features)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(pearson, cmap='coolwarm', center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Feature correlation heatmap (Pearson)')
plt.show()

In [ ]:
# Top correlated feature pairs (both methods).
top_p = relationships.top_correlations(pearson, 'pearson', top_n=15)
top_s = relationships.top_correlations(spearman, 'spearman', top_n=15)
pd.concat([top_p, top_s], ignore_index=True).head(20)

## 6. Association rules

Discretize continuous features into low/med/high bins (+ a win-rate bin), one-hot encode, and mine rules with `mlxtend` apriori. Ranked by lift, filtered to meaningful support/confidence. (Returns empty on very small datasets.)

In [ ]:
rules = relationships.mine_association_rules(features)
print('rules found:', len(rules))
rules.head(20)

## 7. (Optional) Write all artifacts to disk

Equivalent to running `python run_all.py` — writes the CSVs and PNGs into `outputs/`.

In [ ]:
arche = archetypes.run_archetypes(features=features, outdir='outputs', k=K)
rel = relationships.run_relationships(features=features, outdir='outputs')
print('archetype files:', arche['files'])
print('relationship files:', rel['files'])